# Hello World: Training `neural-lam` on DANRA

This notebook provides a **beginner-friendly, end-to-end walkthrough** for running a minimal model training pipeline in `neural-lam` using a small, public DANRA dataset.

It covers:
1. Environment setup (CPU-safe)
2. Data preprocessing with `mllam-data-prep`
3. Graph generation (single-level, for speed)
4. Training for 1 epoch on CPU
5. Evaluation and example predictions
6. Scaling tips for bigger runs

---

### Prerequisites

> **Important:** This notebook is designed to be run from inside a **local clone** of [`mllam/neural-lam`](https://github.com/mllam/neural-lam).
>
> - All paths are **relative to the repository root**.
> - Run all commands from the repo root unless otherwise noted.
> - An internet connection is required for the first data-prep step (data is streamed from a public ECMWF object store).
> - A GPU is **not required**: all commands in this notebook are CPU-safe.
> - Estimated first-run time: ~5–20 minutes depending on internet speed and CPU (dominated by data preprocessing).

In [ ]:
# Verify we are at the repo root (should see neural_lam/, docs/, tests/, etc.)
import os
print("Current directory:", os.getcwd())
print("Repo contents:", [f for f in os.listdir('.') if not f.startswith('.')])

## 1. Environment Setup

You have two options for installation:

**Option A (recommended): `uv`** — if you followed the README setup:
```bash
uv venv --no-project
uv pip install torch --index-url https://download.pytorch.org/whl/cpu
uv pip install .
```

**Option B: `pip`** — run the cell below if you haven't installed yet:

In [ ]:
import sys

# CPU-safe torch install (skip if you already have torch installed)
!{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cpu --quiet

# Install neural-lam and mllam-data-prep from the repo root
!{sys.executable} -m pip install -e . --quiet
!{sys.executable} -m pip install mllam-data-prep --quiet

In [ ]:
# Verify the install
import neural_lam
print("neural-lam version:", neural_lam.__version__)

## 2. Data Configuration & Preprocessing

We use the **DANRA 100m winds example** already in this repo at:
```
tests/datastore_examples/mdp/danra_100m_winds/
  ├── config.yaml          ← neural-lam configuration
  └── danra.datastore.yaml ← mllam-data-prep datastore config
```

This example uses a **cropped DANRA dataset** (~100×100 grid points, ~10 days in April 2022) served from a public ECMWF object store — no local data download is required.

The `mllam-data-prep` command reads the datastore config, fetches the data, and writes a processed `.zarr` archive to disk.

> **Note:** This will download and process approximately 60–120 MB of data. It may take a few minutes on the first run.

In [ ]:
# Run mllam-data-prep to create the processed DANRA .zarr dataset
# Output will appear next to danra.datastore.yaml as danra.datastore.zarr
!python -m mllam_data_prep \
    --config tests/datastore_examples/mdp/danra_100m_winds/danra.datastore.yaml

In [ ]:
# Confirm the .zarr dataset was created
import os
zarr_path = "tests/datastore_examples/mdp/danra_100m_winds/danra.datastore.zarr"
if os.path.exists(zarr_path):
    print("✅ Data preprocessing complete. zarr dataset found at:", zarr_path)
else:
    print("❌ zarr dataset not found — check the output above for errors.")

## 3. Graph Generation

`neural-lam` uses a **graph** to define the message-passing structure of the GNN. Different graph types exist:

| Graph type  | Command flag                    | Use case               |
|-------------|----------------------------------|------------------------|
| L1-LAM      | `--name 1level --levels 1`       | Quick demo (this notebook) |
| GC-LAM      | `--name multiscale`              | Standard multi-scale model |
| Hi-LAM      | `--name hierarchical --hierarchical` | Hierarchical model (production) |

For this Hello World example we use the **L1 (single-level) graph** — the lightest option, ideal for CPU.

In [ ]:
# Generate the single-level graph for fast CPU execution
# Graph files are stored in tests/datastore_examples/mdp/danra_100m_winds/graphs/1level/
!python -m neural_lam.create_graph \
    --config_path tests/datastore_examples/mdp/danra_100m_winds/config.yaml \
    --name 1level \
    --levels 1

In [ ]:
# Confirm the graph was created
import glob
graph_files = glob.glob(
    "tests/datastore_examples/mdp/danra_100m_winds/graphs/1level/**",
    recursive=True
)
print(f"Graph files created ({len(graph_files)}):", graph_files)

## 4. Training on CPU

We train the `graph_lam` model for **1 epoch** using the L1 graph. Key flags used here:

| Flag | Value | Why |
|------|-------|-----|
| `--model` | `graph_lam` | Simplest model — compatible with non-hierarchical graphs |
| `--graph` | `1level` | Must match the graph name used in the previous step |
| `--epochs` | `1` | Minimal run — just to verify the pipeline works end-to-end |
| `--processor_layers` | `2` | Reduced from default (4) for CPU |
| `--ar_steps_train` | `1` | Unroll 1 time-step during training — reduces memory and compute |
| `--ar_steps_eval` | `1` | Also 1 step for the val pass within this demo run |

**About logging:** By default, `neural-lam` logs via [Weights & Biases](https://wandb.ai). To suppress W&B upload during this demo, we set `WANDB_MODE=offline` so metrics are saved locally without requiring a W&B account or login.

In [ ]:
import os
# Set W&B to offline mode for this demo (no account required)
# Remove or change to WANDB_MODE=online to enable cloud logging
os.environ["WANDB_MODE"] = "offline"

!python -m neural_lam.train_model \
    --config_path tests/datastore_examples/mdp/danra_100m_winds/config.yaml \
    --model graph_lam \
    --graph 1level \
    --epochs 1 \
    --processor_layers 2 \
    --ar_steps_train 1 \
    --ar_steps_eval 1 \
    --num_workers 0

In [ ]:
# Find the checkpoint saved during training
import glob
ckpts = glob.glob("saved_models/**/*.ckpt", recursive=True)
print("Checkpoints found:", ckpts)

## 5. Evaluation & Visualization

Evaluation reuses the same `train_model` command with `--eval test` and the `--load` flag pointing to the checkpoint from training.

Example predictions are plotted automatically and saved by the logger. Set `--n_example_pred` to control how many prediction plots to produce.

In [ ]:
# Pick the checkpoint
import glob
ckpts = glob.glob("saved_models/**/*.ckpt", recursive=True)

if not ckpts:
    print("No checkpoint found — make sure the training cell above completed without errors.")
else:
    ckpt_path = ckpts[0]  # use the first (or most recent) checkpoint
    print("Using checkpoint:", ckpt_path)

In [ ]:
import os
os.environ["WANDB_MODE"] = "offline"

!python -m neural_lam.train_model \
    --config_path tests/datastore_examples/mdp/danra_100m_winds/config.yaml \
    --model graph_lam \
    --graph 1level \
    --eval test \
    --load {ckpt_path} \
    --n_example_pred 2 \
    --ar_steps_eval 4 \
    --num_workers 0

## 6. Scaling Tips & Next Steps

### 🔁 Use a larger / hierarchical graph

For a production-quality run, switch to the hierarchical Hi-LAM graph:

```bash
python -m neural_lam.create_graph \
    --config_path <your_config.yaml> \
    --name hierarchical \
    --hierarchical

python -m neural_lam.train_model \
    --config_path <your_config.yaml> \
    --model hi_lam \
    --graph hierarchical \
    --epochs 200
```

### ⚡ Enable GPU training

Simply replace the CPU-only `torch` install with the CUDA variant and `--devices` will auto-detect your GPU:
```bash
pip install torch --index-url https://download.pytorch.org/whl/cu121
```

### 📦 Use larger / full DANRA data

Modify `danra.datastore.yaml` to extend the `coord_ranges.time` window, or point `inputs[*].path` at a larger dataset. See the [mllam-data-prep README](https://github.com/mllam/mllam-data-prep) for full configuration options.

For large datasets (≥10 GB), use parallel preprocessing:
```bash
python -m mllam_data_prep \
    --config <your_datastore.yaml> \
    --dask-distributed-local-core-fraction 0.5
```

### 📊 Enable cloud logging with W&B

Remove the `WANDB_MODE=offline` line (or set it to `online`) and optionally set `--logger wandb --logger-project <your-project-name>`.

### 🔗 Further reading

- [neural-lam README](https://github.com/mllam/neural-lam)
- [mllam-data-prep](https://github.com/mllam/mllam-data-prep)
- [Graph-based Neural Weather Prediction (NeurIPS 2023)](https://arxiv.org/abs/2309.17370)
- [Probabilistic Weather Forecasting with Hierarchical GNNs (NeurIPS 2024)](https://arxiv.org/abs/2406.04759)